# NovaHiring — E2E Test Sistema de Entrevista Híbrido

Este notebook ejercita los **5 comportamientos nuevos** del `DialogueManager` contra la API real:

| Escenario | Candidato | Comportamiento |  
|-----------|-----------|----------------|
| `hibrido_directo` | Sofía Delgado | Todas las respuestas se completan en el primer turno |
| `hibrido_seguimiento` | Carlos Rivas | D1/D3/D5 vagas → el LLM pide follow-up → segunda respuesta cierra |
| `hibrido_clarificacion` | Elena Martínez | D1/D4 hacen preguntas → el LLM responde y re-pregunta |
| `hibrido_correccion` | Sofía Delgado | D2 vaga → corrección solicitada → respuesta completa |
| `hibrido_abandono` | Elena Martínez | D1/D2 respondidas → D3 abandono → sesión cerrada |

## Requisitos antes de correr

```bash
docker compose up -d          # PostgreSQL + Redis
uv run python seed.py         # Caso de referencia + prompts
uv run fastapi dev main.py    # API en :8000
```

El proveedor activo puede ser Anthropic o OpenAI — el dataset funciona con ambos.

**Costo estimado (Anthropic):** ~$0.05–0.15 por escenario completo (8 llamadas dialogue_conductor + 8 evaluator).

> Para limpiar sesiones entre ejecuciones: `make clean-sessions`

In [ ]:
import json
import os
import sys
import time
from decimal import Decimal
from pathlib import Path

import httpx
from dotenv import load_dotenv

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / '.env')

DATASET_PATH = Path().resolve() / 'e2e_hybrid_dataset.json'
BASE_URL = 'http://localhost:8000'

dataset = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
JOB_ID = dataset['job_id']
MAX_TURNS = dataset['max_turns_per_dimension']

# Leer API key del .env — candidates key para /interviews/*, admin para todo
_api_key = os.getenv('API_KEY_CANDIDATES') or os.getenv('API_KEY_ADMIN') or ''
_headers = {'X-API-Key': _api_key} if _api_key else {}
if _api_key:
    print(f'  API key: {_api_key[:8]}... (cargada de .env)')
else:
    print('  Sin API key — auth desactivada (modo dev)')

client = httpx.Client(base_url=BASE_URL, timeout=120.0, headers=_headers)

print(f'✓ Dataset cargado: {len(dataset["scenarios"])} escenarios')
print(f'  Job ID: {JOB_ID}')
print(f'  Máximo de turnos por dimensión: {MAX_TURNS}')
for s in dataset['scenarios']:
    total_turns = sum(len(d['turns']) for d in s['dimensions'])
    print(f'  • {s["id"]:30s} {s["candidate_id"]:30s} {total_turns} mensajes a enviar')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def get_session_state(session_id: str) -> dict:
    r = client.get(f'/api/v1/interviews/sessions/{session_id}')
    r.raise_for_status()
    return r.json()


def start_session(candidate_id: str) -> dict:
    """Crea una sesión nueva o retoma la activa existente."""
    r = client.post('/api/v1/interviews/sessions', json={
        'job_id': JOB_ID,
        'candidate_id': candidate_id,
    })
    if r.status_code == 409:
        detail = r.json().get('detail', {})
        session_id = detail.get('session_id')
        state = get_session_state(session_id)
        if state.get('status') == 'completed':
            raise RuntimeError(
                f'Sesión ya completada para {candidate_id}. '
                f'Ejecuta make clean-sessions para limpiar.'
            )
        print(f'  ⚠ Sesión activa existente retomada: {session_id}')
        return {'session_id': session_id, 'existing': True}
    r.raise_for_status()
    data = r.json()
    return {'session_id': data['session_id'], 'existing': False}


def send_message(session_id: str, content: str) -> dict:
    """Envía un mensaje y devuelve la respuesta. Nunca lanza excepción por estado 4xx."""
    r = client.post(
        f'/api/v1/interviews/sessions/{session_id}/message',
        json={'content': content},
    )
    if r.status_code in (410, 409, 422):
        return {'_http_status': r.status_code, '_body': r.json()}
    r.raise_for_status()
    return r.json()


def dim_from_response(resp: dict) -> str | None:
    """Extrae el dimension_id de la respuesta. None si sesión cerrada."""
    nq = resp.get('next_question')
    if not nq:
        return None
    return nq.get('dimension_id')


print('✓ Helpers definidos')

In [ ]:
# ── Verificar servidor ────────────────────────────────────────────────────────
try:
    r = client.get('/health')
    r.raise_for_status()
    print(f'✓ Servidor OK → {r.json()}')
except httpx.ConnectError:
    print('✗ El servidor no está corriendo.')
    print('  Ejecuta: make start-dev')
    raise SystemExit(1)

In [ ]:
# ── Motor principal v2: sigue el estado real del servidor ─────────────────────
#
# DISEÑO ANTERIOR (roto): iteraba dimension por dimension del dataset sin
# importar qué respondía el servidor. Si el LLM pedía un 3er follow-up en D1,
# el notebook ya había enviado las respuestas de D2 y D3 como si fueran D1.
#
# DISEÑO NUEVO: el servidor manda. Después de cada mensaje, leemos
# next_question.dimension_id para saber en qué dimensión está el servidor,
# y enviamos el siguiente turno de ESA dimensión.

def run_scenario(scenario: dict) -> dict:
    scenario_id = scenario['id']
    candidate_id = scenario['candidate_id']

    print(f"\n{'='*65}")
    print(f"Escenario : {scenario_id}")
    print(f"Candidato : {candidate_id}")
    print(f"Descripción: {scenario['descripcion'][:80]}...")
    print(f"{'='*65}")

    session_info = start_session(candidate_id)
    session_id = session_info['session_id']
    print(f'  ✓ Sesión: {session_id}')

    # Índice de turnos por dimensión: dim_id → [turn0, turn1, ...]
    turns_by_dim = {d['dimension_id']: list(d['turns']) for d in scenario['dimensions']}
    # Puntero: cuántos turnos ya enviamos por dimensión
    sent_idx: dict[str, int] = {dim_id: 0 for dim_id in turns_by_dim}

    turn_records = []
    evaluation_result = None
    final_status = 'active'

    # Dimensión inicial: la primera del dataset
    server_dim_id = scenario['dimensions'][0]['dimension_id']

    # Límite de seguridad para no quedar en loop infinito
    MAX_MESSAGES = 40

    for _guard in range(MAX_MESSAGES):
        # ── elegir qué turno enviar ───────────────────────────────────────────
        dim_turns = turns_by_dim.get(server_dim_id, [])
        idx = sent_idx.get(server_dim_id, 0)

        if idx >= len(dim_turns):
            # El servidor está en una dimensión de la que no tenemos más turnos.
            # El servidor forzará completar tras MAX_TURNS — no enviamos nada.
            print(f'  ⚠ Sin más turnos para {server_dim_id} en el dataset. El servidor forzará completar.')
            break

        turn = dim_turns[idx]
        sent_idx[server_dim_id] = idx + 1
        content = turn['content']
        expected_complete = turn['is_complete_esperado']
        expected_intent = turn['intent_esperado']

        label = f'[{server_dim_id} turno {idx+1}/{len(dim_turns)}]'
        print(f'  Enviando {label}...', end=' ', flush=True)
        t0 = time.monotonic()

        resp = send_message(session_id, content)
        elapsed = time.monotonic() - t0

        # ── clasificar respuesta ──────────────────────────────────────────────
        http_err = resp.get('_http_status')
        if http_err:
            outcome = f'HTTP_{http_err}'
            observed_complete = False
            print(f'⚠ HTTP {http_err} ({elapsed:.1f}s)')
        elif resp.get('evaluation_result'):
            evaluation_result = resp['evaluation_result']
            outcome = 'evaluacion_completa'
            observed_complete = True
            final_status = 'completed'
            print(f'✓ EVALUACIÓN COMPLETA ({elapsed:.1f}s)')
        else:
            resp_dim = dim_from_response(resp)
            if resp_dim is None:
                outcome = 'sesion_cerrada'
                observed_complete = False
                final_status = 'abandoned'
                print(f'✓ sesión cerrada/abandono ({elapsed:.1f}s)')
            elif resp_dim != server_dim_id:
                # El servidor avanzó a una nueva dimensión
                outcome = f'avance→{resp_dim}'
                observed_complete = True
                server_dim_id = resp_dim          # ← seguimos al servidor
                print(f'✓ avance a {resp_dim} ({elapsed:.1f}s)')
            else:
                outcome = 'misma_dimension'
                observed_complete = False
                print(f'✓ seguimiento/clarif/correccion ({elapsed:.1f}s)')

        match = observed_complete == expected_complete
        status_icon = '✓' if match else '✗'
        print(f'    {status_icon} is_complete: esperado={expected_complete} observado={observed_complete} | {turn["nota"][:60]}')

        turn_records.append({
            'dimension_id': server_dim_id if observed_complete else list(turns_by_dim.keys())[
                list(sent_idx.keys()).index(server_dim_id) if server_dim_id in sent_idx else 0
            ],
            'dimension_id': server_dim_id,
            'turn_idx': idx + 1,
            'intent_esperado': expected_intent,
            'is_complete_esperado': expected_complete,
            'outcome': outcome,
            'is_complete_observado': observed_complete,
            'match': match,
            'elapsed_s': round(elapsed, 2),
            'nota': turn['nota'],
        })

        if outcome in ('sesion_cerrada', 'evaluacion_completa') or http_err:
            break

    # ── estado final ──────────────────────────────────────────────────────────
    try:
        session_state = get_session_state(session_id)
        final_status = session_state.get('status', final_status)
    except Exception:
        pass

    matches = sum(1 for t in turn_records if t['match'])
    total = len(turn_records)
    print(f'  Turnos correctos: {matches}/{total}')
    print(f'  Estado final sesión: {final_status}')
    if evaluation_result:
        print(f'  Score final: {evaluation_result["weighted_score"]} — {evaluation_result["resultado"]}')

    return {
        'scenario_id': scenario_id,
        'candidate_id': candidate_id,
        'session_id': session_id,
        'turn_records': turn_records,
        'matches': matches,
        'total_turns': total,
        'evaluation_result': evaluation_result,
        'final_status': final_status,
        'expected': scenario['expected_result'],
    }


print('✓ Motor v2 definido (server-state-driven)')

In [ ]:
# ── Ejecutar todos los escenarios ─────────────────────────────────────────────
# Los escenarios están ordenados para respetar la restricción de sesión única por candidato:
# hibrido_directo (Sofía) → completado antes de hibrido_correccion (Sofía)
# hibrido_clarificacion (Elena) → completado antes de hibrido_abandono (Elena)

all_results = []

for scenario in dataset['scenarios']:
    result = run_scenario(scenario)
    all_results.append(result)

print(f"\n{'='*65}")
print(f'RESUMEN: {len(all_results)} escenarios ejecutados')

In [ ]:
# ── Tabla de resultados por escenario ─────────────────────────────────────────

print(f"\n{'='*80}")
print('RESULTADOS POR ESCENARIO')
print(f"{'='*80}")
print(f"{'Escenario':<28} {'Candidato':<20} {'Turnos OK':>9} {'Estado final':<14} {'Score':>7} {'OK?'}")
print('-' * 80)

INTENT_ICONS = {
    'completed': '✅',
    'abandoned': '🚪',
    'active': '🔄',
}

total_matches = 0
total_turns_all = 0

for r in all_results:
    pct = r['matches'] / r['total_turns'] * 100 if r['total_turns'] else 0
    score_str = r['evaluation_result']['weighted_score'] if r['evaluation_result'] else '—'
    expected_status = r['expected'].get('resultado') or r['expected'].get('session_status_final', '?')
    icon = INTENT_ICONS.get(r['final_status'], '?')

    # Verificar resultado vs esperado
    if r['expected'].get('resultado') == 'abandoned':
        ok = '✓' if r['final_status'] == 'abandoned' else '✗'
    elif r['expected'].get('resultado') == 'APTO' and r['evaluation_result']:
        real_score = Decimal(r['evaluation_result']['weighted_score'])
        min_score = Decimal(r['expected'].get('min_weighted_score', '0'))
        ok = '✓' if real_score >= min_score and r['evaluation_result']['resultado'] == 'APTO' else '✗'
    else:
        ok = '?'

    print(f"{r['scenario_id']:<28} {r['candidate_id'].split('-', 2)[-1][:20]:<20} "
          f"{r['matches']}/{r['total_turns']:>2} ({pct:4.0f}%) "
          f"{icon} {r['final_status']:<12} {score_str:>7} {ok}")

    total_matches += r['matches']
    total_turns_all += r['total_turns']

print('-' * 80)
global_pct = total_matches / total_turns_all * 100 if total_turns_all else 0
print(f"{'TOTAL':<28} {'':20} {total_matches}/{total_turns_all:>2} ({global_pct:4.0f}%)")
print(f"\n  Turnos correctos: {total_matches}/{total_turns_all}")
print(f"  (Un turno es 'correcto' cuando is_complete observado == is_complete esperado)")

In [ ]:
# ── Detalle por turno para cada escenario ─────────────────────────────────────

OUTCOME_LABELS = {
    'evaluacion_completa': '🏁 evaluación completa',
    'sesion_cerrada':      '🚪 sesión cerrada',
    'misma_dimension':     '🔄 mismo dim (follow-up)',
}

for r in all_results:
    print(f"\n── {r['scenario_id']} ({r['candidate_id']}) ──")
    print(f"  {'Dim':>4} {'T':>2} {'intent esp.':<22} {'complete esp.':>13} {'outcome':<28} {'OK?'}")
    print(f"  {'-'*4} {'-'*2} {'-'*22} {'-'*13} {'-'*28} {'-'*3}")
    for t in r['turn_records']:
        outcome_label = OUTCOME_LABELS.get(t['outcome'], t['outcome'])
        ok = '✓' if t['match'] else '✗'
        print(f"  {t['dimension_id']:>4} {t['turn_idx']:>2} {t['intent_esperado']:<22} "
              f"{'complete' if t['is_complete_esperado'] else 'incomplete':>13} "
              f"{outcome_label:<28} {ok}")

    if r['evaluation_result']:
        er = r['evaluation_result']
        print(f"  → Score: {er['weighted_score']} / 5.00 — {er['resultado']}")
    elif r['final_status'] == 'abandoned':
        exp_last = r['expected'].get('last_completed_dimension', '?')
        print(f"  → Sesión abandonada. Última dimensión completada: {exp_last}")

In [ ]:
# ── Análisis de comportamientos híbridos específicos ──────────────────────────

print('VERIFICACIÓN DE COMPORTAMIENTOS HÍBRIDOS')
print('='*60)

# 1. Seguimiento: ¿D1/D3/D5 generaron follow-up?
seguimiento = next((r for r in all_results if r['scenario_id'] == 'hibrido_seguimiento'), None)
if seguimiento:
    followup_dims = [
        t['dimension_id'] for t in seguimiento['turn_records']
        if t['outcome'] == 'misma_dimension'
    ]
    expected_followups = seguimiento['expected'].get('expected_followup_dimensions', [])
    ok = all(d in followup_dims for d in expected_followups)
    print(f"\n1. Seguimiento (hibrido_seguimiento)")
    print(f"   Follow-ups esperados: {expected_followups}")
    print(f"   Follow-ups observados en: {list(dict.fromkeys(followup_dims))}")
    print(f"   Resultado: {'✓ CORRECTO' if ok else '✗ INCORRECTO'}")

# 2. Clarificación: ¿D1/D4 generaron turno de clarificación?
clarif = next((r for r in all_results if r['scenario_id'] == 'hibrido_clarificacion'), None)
if clarif:
    clarif_dims = [
        t['dimension_id'] for t in clarif['turn_records']
        if t['intent_esperado'] == 'asking_clarification' and t['outcome'] == 'misma_dimension'
    ]
    expected_clarifs = clarif['expected'].get('expected_clarification_dimensions', [])
    ok = len(clarif_dims) >= len(expected_clarifs)
    print(f"\n2. Clarificación (hibrido_clarificacion)")
    print(f"   Clarificaciones esperadas en: {expected_clarifs}")
    print(f"   Clarificaciones detectadas en: {clarif_dims}")
    print(f"   Resultado: {'✓ CORRECTO' if ok else '✗ INCORRECTO'}")

# 3. Corrección: ¿D2 tiene 3 turnos (vago → corrección → completo)?
correccion = next((r for r in all_results if r['scenario_id'] == 'hibrido_correccion'), None)
if correccion:
    d2_turns = [t for t in correccion['turn_records'] if t['dimension_id'] == 'D2']
    correction_turn = next((t for t in d2_turns if t['intent_esperado'] == 'requesting_correction'), None)
    expected_dims = correccion['expected'].get('expected_correction_dimensions', [])
    ok = correction_turn is not None and correction_turn['outcome'] == 'misma_dimension'
    print(f"\n3. Corrección (hibrido_correccion)")
    print(f"   Dimensiones con corrección esperadas: {expected_dims}")
    print(f"   Turnos D2: {len(d2_turns)} turno(s)")
    if correction_turn:
        print(f"   Turno de corrección detectado: {correction_turn['outcome']}")
    print(f"   Resultado: {'✓ CORRECTO' if ok else '✗ INCORRECTO'}")

# 4. Abandono: ¿sesión cerrada con status=abandoned?
abandono = next((r for r in all_results if r['scenario_id'] == 'hibrido_abandono'), None)
if abandono:
    ok = abandono['final_status'] == 'abandoned'
    print(f"\n4. Abandono (hibrido_abandono)")
    print(f"   Estado final esperado: abandoned")
    print(f"   Estado final observado: {abandono['final_status']}")
    print(f"   Resultado: {'✓ CORRECTO' if ok else '✗ INCORRECTO'}")

# 5. Directo: ¿todas las dimensiones completadas en turno 1?
directo = next((r for r in all_results if r['scenario_id'] == 'hibrido_directo'), None)
if directo:
    all_first_turn_complete = all(
        t['turn_idx'] == 1 and t['is_complete_observado']
        for t in directo['turn_records']
        if t['is_complete_esperado']
    )
    score = directo['evaluation_result']['weighted_score'] if directo['evaluation_result'] else '—'
    min_score = directo['expected'].get('min_weighted_score', '0')
    score_ok = directo['evaluation_result'] and Decimal(score) >= Decimal(min_score)
    print(f"\n5. Directo (hibrido_directo)")
    print(f"   Todas las dimensiones en turno 1: {'✓' if all_first_turn_complete else '✗'}")
    print(f"   Score final: {score} (mínimo esperado: {min_score})")
    print(f"   Score OK: {'✓ CORRECTO' if score_ok else '✗ INCORRECTO'}")

In [ ]:
# ── Ranking de candidatos que llegaron a evaluación ───────────────────────────

evaluated = [
    r for r in all_results
    if r['evaluation_result'] is not None
]

if not evaluated:
    print('Ningún escenario completó la evaluación.')
else:
    ranked = sorted(evaluated, key=lambda r: Decimal(r['evaluation_result']['weighted_score']), reverse=True)

    print(f"\n{'='*70}")
    print('RANKING — Escenarios que llegaron a evaluación completa')
    print(f"{'='*70}")
    print(f"{'Pos':<4} {'Escenario':<28} {'Candidato':<20} {'Score':>7} {'Resultado'}")
    print('-' * 70)

    for i, r in enumerate(ranked, start=1):
        er = r['evaluation_result']
        cand_short = '-'.join(r['candidate_id'].split('-')[2:]).replace('-', ' ').title()
        print(f"{i:<4} {r['scenario_id']:<28} {cand_short:<20} {float(er['weighted_score']):>6.2f} {er['resultado']}")

    print(f"\nTotal evaluaciones completadas: {len(evaluated)}/{len(all_results)}")

    # Sesiones abandonadas
    abandoned = [r for r in all_results if r['final_status'] == 'abandoned']
    if abandoned:
        print(f"\nSesiones abandonadas: {len(abandoned)}")
        for r in abandoned:
            print(f"  • {r['scenario_id']} ({r['candidate_id']}) — sin evaluación generada")

In [ ]:
# ── Verificación en base de datos (consultas de referencia) ───────────────────

session_ids = [r['session_id'] for r in all_results]

print('Para inspeccionar manualmente en la base de datos:')
print()
print('  docker exec -it test-nova-hiring-postgres-1 psql -U nova -d nova_hiring')
print()
print('-- Estado de todas las sesiones de este notebook:')
ids_list = "','".join(session_ids)
print(f"SELECT id, candidate_id, status, created_at")
print(f"FROM chat_sessions WHERE id IN ('{ids_list}');")
print()
print('-- Mensajes de la sesión de corrección (ver turnos de D2):')
correccion_r = next((r for r in all_results if r['scenario_id'] == 'hibrido_correccion'), None)
if correccion_r:
    print(f"SELECT sequence_number, role, LEFT(content, 80) AS content_preview")
    print(f"FROM messages WHERE session_id = '{correccion_r['session_id']}'")
    print(f"ORDER BY sequence_number;")
print()
print('-- Evaluaciones generadas en esta sesión de E2E:')
print("SELECT e.id, c.nombre, e.weighted_score, e.resultado, e.created_at")
print("FROM evaluations e JOIN candidates c ON c.id = e.candidate_id")
print("ORDER BY e.created_at DESC LIMIT 5;")
print()
print('-- Llamadas al dialogue_conductor (nuevas en el sistema híbrido):')
print("SELECT prompt_name, input_tokens, output_tokens, latency_ms, validation_passed, created_at")
print("FROM ai_call_logs")
print("WHERE prompt_name = 'interview_dialogue_conductor'")
print("ORDER BY created_at DESC LIMIT 20;")

In [ ]:
# ── Inspeccionar conversaciones reales en la DB ───────────────────────────────
# Corre esta celda DESPUÉS de run_scenario para ver los mensajes reales
# que el servidor guardó, conversación por conversación.

import subprocess, textwrap

CONTAINER = 'test-nova-hiring-postgres-1'

def psql(sql: str) -> str:
    result = subprocess.run(
        ['docker', 'exec', CONTAINER, 'psql', '-U', 'nova', '-d', 'nova_hiring', '-c', sql],
        capture_output=True, text=True
    )
    return result.stdout or result.stderr


# ── 1. Estado de las sesiones de este notebook ────────────────────────────────
if 'all_results' in dir() and all_results:
    ids_sql = "','".join(r['session_id'] for r in all_results)
    print('── SESIONES ──────────────────────────────────────────────────')
    print(psql(f"""
        SELECT cs.id, c.nombre, cs.status,
               (SELECT COUNT(*) FROM messages m WHERE m.session_id = cs.id) AS n_msgs,
               cs.created_at::time AS hora
        FROM chat_sessions cs
        JOIN candidates c ON c.id = cs.candidate_id
        WHERE cs.id IN ('{ids_sql}')
        ORDER BY cs.created_at;
    """))
else:
    print('⚠ all_results no disponible. Corre la celda 5 primero.')

In [ ]:
# ── 2. Conversación completa por escenario ────────────────────────────────────
# Muestra todos los mensajes de cada sesión, en orden, con el texto truncado.

if 'all_results' not in dir() or not all_results:
    print('⚠ all_results no disponible.')
else:
    for r in all_results:
        sid = r['session_id']
        print(f"\n{'='*70}")
        print(f"ESCENARIO: {r['scenario_id']}  |  {r['candidate_id']}")
        print(f"Sesión:    {sid}")
        print(f"Estado:    {r['final_status']}")
        print(f"{'='*70}")

        rows = psql(f"""
            SELECT sequence_number AS seq,
                   role,
                   LEFT(content, 120) AS texto
            FROM messages
            WHERE session_id = '{sid}'
            ORDER BY sequence_number;
        """)
        print(rows)

In [ ]:
# ── 3. Llamadas a la IA por sesión ────────────────────────────────────────────
# Cuántas veces se llamó al dialogue_conductor y al response_evaluator.

if 'all_results' not in dir() or not all_results:
    print('⚠ all_results no disponible.')
else:
    ids_sql = "','".join(r['session_id'] for r in all_results)
    print('── LLAMADAS A LA IA ──────────────────────────────────────────')
    print(psql(f"""
        SELECT l.session_id,
               l.prompt_name,
               COUNT(*) AS n_llamadas,
               SUM(l.input_tokens)  AS tokens_in,
               SUM(l.output_tokens) AS tokens_out,
               ROUND(AVG(l.latency_ms)) AS avg_ms
        FROM ai_call_logs l
        WHERE l.session_id IN ('{ids_sql}')
        GROUP BY l.session_id, l.prompt_name
        ORDER BY l.session_id, l.prompt_name;
    """))

## ¿Qué acabó de pasar?

```
Notebook (por turno)                Servidor FastAPI              Anthropic/OpenAI
────────────────────                ────────────────              ────────────────
POST /message (turno N)  ──────►   Guarda mensaje en DB
                                   DialogueManager.process_turn()
                                   ai_client.call(dialogue_conductor) ──► LLM
                                                                     ◄─── {intent, is_complete, response}
                         ◄──────   Respuesta + next_question (mismo dim o siguiente)

Si is_complete=True en D8: ──────► _run_evaluation_pipeline()
                                   Para cada D1..D8:
                                     ai_client.call(response_evaluator) ──► LLM
                                                                        ◄─── {score, justif}
                                   Scorer.calculate()  (Python puro)
                                   INSERT evaluations
                         ◄──────   evaluation_result con score final
```

**El LLM hizo dos tipos de trabajo:**
- `interview_dialogue_conductor`: clasificó intent y decidió si la respuesta era suficiente
- `interview_response_evaluator`: puntuó las respuestas bloqueadas (igual que antes)

**Python mantuvo todo lo demás:** orden de preguntas, límite de 3 turnos, criterios KO, fórmula de puntuación.